# ml_server_embedding

In [ ]:
"""
MLServer Embedding Module
Provides embeddings via MLServer inference endpoint.

Usage in notebooks:
    from ml_server_embedding import get_embeddings

    # Use default model
    embeddings = get_embeddings()

    # Or specify custom model
    embeddings = get_embeddings(model="custom-model-name")
"""

from langchain_core.embeddings import Embeddings
from pydantic import BaseModel
from typing import Any, List, Mapping, Optional
import requests
import numpy as np


# Default configuration
DEFAULT_MODEL = "all-mpnet-base--fccb5"
DEFAULT_BASE_URL = ""


class MLServerEmbedding(Embeddings, BaseModel):
    max_batch_size: int = 32
    url: str

    """ public """

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        text_num = len(texts)
        result = []
        for i in range(0, text_num, self.max_batch_size):
            result += self._get_embedding(texts[i: i + self.max_batch_size])
        return result

    def embed_query(self, text: str) -> List[float]:
        return self._get_embedding([text])[0]

    """ private """

    @staticmethod
    def _wrap_payload(text_list):
        return {
            "inputs": [
                {
                    "name": "input",
                    "shape": [len(text_list)],
                    "datatype": "str",
                    "data": text_list
                }
            ]
        }

    @staticmethod
    def _parse_response(response):
        if response.status_code != 200:
            raise Exception(response)
        outputs = response.json()["outputs"][0]
        return np.array(outputs["data"]).reshape(outputs["shape"]).tolist()

    def _get_embedding(self, text_list: List[str]) -> List[List[float]]:
        return self._parse_response(requests.post(url=self.url,
                                                  json=self._wrap_payload(text_list),
                                                  headers={"Content-Type": "application/json"},
                                                  params={}))


def get_embeddings(model: str = DEFAULT_MODEL, base_url: str = DEFAULT_BASE_URL, max_batch_size: int = 32):
    """
    Create and return a configured MLServerEmbedding instance.

    Args:
        model (str): Model name to use. Default: "all-mpnet-base--fccb5"
        base_url (str): Base URL for the MLServer endpoint. Default: internal URL
        max_batch_size (int): Maximum batch size for embedding requests. Default: 32

    Returns:
        MLServerEmbedding: Configured embedding instance

    Example:
        >>> embeddings = get_embeddings()
        >>> embeddings = get_embeddings(model="custom-model-v2")
        >>> vector = embeddings.embed_query("Hello world")
    """
    # Construct the full URL
    embedding_url = f"{base_url}/{model}/v2/models/{model}/infer"

    # Create and return embedding instance
    return MLServerEmbedding(url=embedding_url, max_batch_size=max_batch_size)


# For backward compatibility - create default instance
embeddings = get_embeddings()